# Pratica 10 — Word embedding

**Corso:** NLP Classico con Python · **Lezione 10** · **Tempo stimato:** 60-75 minuti
**Notebook di lezione:** `notebooks_clase/clase10_word_embeddings_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Gli esercizi 1 e 2 usano cifre che puoi calcolare a mano; i 3-5 usano l'estratto del Quijote di `datos/gutenberg/` (~33 000 parole: **poche**, e quindi i vettori saranno rumorosi) e la sfida anche il corpus grande di quattro opere (in Colab, carica la cartella `datos/`). È spagnolo antico, non quello di oggi (*esempio illustrativo*).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `gensim`.

---
## Cella 0 — Setup e dati

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("gensim", "gensim")
print("Librerie pronte.")

In [ ]:
import os
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models import Word2Vec, FastText

warnings.filterwarnings("ignore")
random.seed(42)                          # seme fisso
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

def frases_de_texto(texto):
    """Minuscole, taglio in frasi e solo lettere; scarta le frasi di meno di 3 parole."""
    salida = []
    for trozo in re.split(r"[.!?;:]+\s|\n+", texto.lower()):
        palabras = re.findall(r"[^\W\d_]+", trozo)
        if len(palabras) >= 3:
            salida.append(palabras)
    return salida

def vecinos(modelo, palabra, n=6):
    """Le n parole più vicine per coseno."""
    return [w for w, _ in modelo.wv.most_similar(palabra, topn=n)]

def frecuentes(frases, modelo, n=200, largo=4):
    """Le n parole più frequenti (di almeno `largo` lettere) presenti nel vocabolario del modello."""
    cuenta = Counter(w for f in frases for w in f)
    return [w for w, _ in cuenta.most_common() if len(w) >= largo and w in modelo.wv.key_to_index][:n]

print("Funzioni ausiliarie caricate.")

In [ ]:
# datos-es
# Mini-corpus di 4 frasi (spagnolo) per l'esercizio 2 e vocabolario di 9 parole
FRASES_P = [["el", "rey", "ama", "a", "la", "reina"], ["la", "reina", "ama", "al", "rey"],
            ["el", "pastor", "ama", "a", "la", "pastora"], ["la", "pastora", "ama", "al", "pastor"]]
VOCAB_P = ["rey", "reina", "pastor", "pastora", "el", "la", "ama", "a", "al"]
RUTA_PEQUENO = "gutenberg/quijote_cap01-14.txt"
RUTA_GRANDE = "gutenberg/embeddings_es_obras.txt"

In [ ]:
with open(cargar_datos(RUTA_PEQUENO), encoding="utf-8") as f:
    PEQUENO = frases_de_texto(f.read())
print("Estratto piccolo del Quijote: frasi", len(PEQUENO), "| parole", sum(map(len, PEQUENO)))
assert 1000 < len(PEQUENO) < 2000

## Esercizio 1 — La similarità del coseno

Completa `coseno_p(a, b)`: `a·b / (‖a‖ ‖b‖)`, con NumPy. Prima di programmare, a mano: `a = (1, 2, 2)` e `b = (2, 1, 2)` danno `a·b = 2 + 2 + 4 = 8` e `‖a‖ = ‖b‖ = 3`, quindi `cos = 8/9 ≈ 0,889`.

In [ ]:
def coseno_p(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return None    # TODO: prodotto scalare diviso per il prodotto delle norme (np.dot e np.linalg.norm)

# Controllo automatico
_c = coseno_p([1, 2, 2], [2, 1, 2])
assert _c is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_c, 8 / 9), "a mano: 8/9 ≈ 0,889"
assert np.isclose(coseno_p([3, 4], [30, 40]), 1.0) and np.isclose(coseno_p([3, 4], [-4, 3]), 0.0) and np.isclose(coseno_p([3, 4], [-3, -4]), -1.0)
assert np.isclose(coseno_p([1, 2, 2], [2, 1, 2]), coseno_p([2, 4, 4], [1, 1, 1] + np.array([1, 0, 1])))   # non dipende dalla lunghezza
print("Esercizio 1 corretto ✔", round(_c, 3))

## Esercizio 2 — Matrice di co-occorrenza (vettori sparsi)

Completa `matriz_coocurrencia(frases, vocab, ventana=1)`: matrice quadrata di conteggi (riga = parola, colonna = parola di contesto) contando, per ogni parola, quelle a distanza `ventana` o meno **nella stessa frase** (senza contare se stessa nella propria posizione). Si contano solo le parole presenti in `vocab`.

A mano, con `FRASES_P` e finestra 1: `rey` compare in `el rey ama` (vicine `el` e `ama`) e in `ama al rey` (vicina `al`), quindi la sua riga ha 1 in `el`, 1 in `ama` e 1 in `al`, e zeri altrove.

In [ ]:
def matriz_coocurrencia(frases, vocab, ventana=1):
    indice = {p: i for i, p in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)), dtype=int)
    for frase in frases:
        for i, p in enumerate(frase):
            pass    # TODO: per ogni posizione j a distanza <= ventana da i (j diverso da i, dentro la frase), somma 1 in M[indice[p], indice[frase[j]]] se entrambe sono in vocab
    return M

# Controllo automatico
def _cos(a, b):
    return a @ b / np.linalg.norm(a) / np.linalg.norm(b)

_M1 = matriz_coocurrencia(FRASES_P, VOCAB_P, 1)
_ix = {p: i for i, p in enumerate(VOCAB_P)}
assert _M1.sum() > 0, "Completa le righe contrassegnate con TODO"
assert _M1.shape == (9, 9) and (_M1 == _M1.T).all(), "la matrice deve essere quadrata e simmetrica"
assert [int(_M1[_ix["rey"], _ix[w]]) for w in VOCAB_P[4:5] + ["ama", "al", "reina"]] == [1, 1, 1, 0]      # il calcolo a mano
assert np.isclose(_cos(_M1[_ix["rey"]], _M1[_ix["pastor"]]), 1.0)          # stessi contesti con finestra 1
assert np.isclose(_cos(_M1[_ix["rey"]], _M1[_ix["reina"]]), 1 / np.sqrt(15))
_M2 = matriz_coocurrencia(FRASES_P, VOCAB_P, 2)
assert _M2.sum() > _M1.sum() and np.isclose(_cos(_M2[_ix["rey"]], _M2[_ix["reina"]]), 4 / 7)   # finestra maggiore: più co-occorrenze
print("Esercizio 2 corretto ✔", int(_M1.sum()), int(_M2.sum()))

## Esercizio 3 — Addestrare Word2Vec in modo riproducibile

Completa `entrenar_p(frases, sg, window, seed)`: un `Word2Vec` con `vector_size=30`, `min_count=3`, `epochs=5`, `workers=1`, il `seed` dato (con un solo thread, il risultato è lo stesso a ogni esecuzione), e con `sg` e `window` come parametri (`sg=1` skip-gram, `sg=0` CBOW). Si addestra su `PEQUENO` (l'estratto del Quijote: ~33 000 parole, poche).

In [ ]:
def entrenar_p(frases, sg, window, seed):
    return None    # TODO: Word2Vec(frases, vector_size=30, window=window, min_count=3, sg=sg, epochs=5, workers=1, seed=seed)

# Controllo automatico
_a = entrenar_p(PEQUENO, 1, 5, 1)
assert _a is not None, "Completa le righe contrassegnate con TODO"
_b = entrenar_p(PEQUENO, 1, 5, 1)
_c = entrenar_p(PEQUENO, 0, 3, 2)
assert _a.wv.vector_size == 30 and _a.sg == 1 and _c.sg == 0 and _c.window == 3, "rivedi vector_size, sg e window"
assert _a.wv.key_to_index == _b.wv.key_to_index and np.allclose(_a.wv.vectors, _b.wv.vectors), "stessi dati e stesso seme devono dare lo stesso modello (workers=1)"
assert not np.allclose(_a.wv.vectors, entrenar_p(PEQUENO, 1, 5, 2).wv.vectors), "un altro seme deve dare altri vettori"
print("Esercizio 3 corretto ✔", "vocabolario:", len(_a.wv))

## Esercizio 4 — Misurare il rumore: solape tra due modelli

Completa `solape_p(m1, m2, palabras, k=10)`: per ogni parola, la frazione dei suoi `k` vicini più prossimi (usa la funzione `vecinos(modelo, palabra, k)` già definita) che **coincidono** tra i due modelli; restituisci la media. A mano: se `m1` dà `[a, b, c, d]` e `m2` dà `[a, b, x, y]` con `k = 4`, coincidono 2 su 4 → 0,5.

In [ ]:
def solape_p(m1, m2, palabras, k=10):
    valores = []
    for w in palabras:
        pass    # TODO: calcola la frazione di coincidenze tra vecinos(m1, w, k) e vecinos(m2, w, k) e aggiungila a valores
    return float(np.mean(valores)) if valores else None

# Controllo automatico
class _Falso:                                     # modello giocattolo che restituisce vicini fissi
    def __init__(self, lista):
        self.wv = self
        self._lista = lista
    def most_similar(self, palabra, topn=10):
        return [(w, 1.0) for w in self._lista[:topn]]

_f1, _f2 = _Falso(["a", "b", "c", "d"]), _Falso(["a", "b", "x", "y"])
_r = solape_p(_f1, _f2, ["p"], 4)
assert _r is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_r, 0.5) and np.isclose(solape_p(_f1, _f1, ["p", "q"], 4), 1.0), "a mano: 2 su 4 = 0,5"
_m1 = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1)
_m2 = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=2)
_frec = frecuentes(PEQUENO, _m1, 50)
_mismo, _otro = solape_p(_m1, _m1, _frec, 10), solape_p(_m1, _m2, _frec, 10)
assert np.isclose(_mismo, 1.0) and 0 <= _otro < 1
print("Esercizio 4 corretto ✔", "solape tra semi nell'estratto piccolo:", round(_otro, 3))

## Esercizio 5 — fastText e parole fuori vocabolario

Si addestra un `FastText` su `PEQUENO` (cella qui sotto). Completa `vector_de(modelo, palabra)`: restituisce la coppia `(esta_en_vocabulario, vector)`, dove `esta_en_vocabulario` dice se `palabra` è in `modelo.wv.key_to_index` e `vector` è `modelo.wv[palabra]`. Poi verifica che `Word2Vec` solleva un `KeyError` con il refuso `caballeroo` e che fastText **sì** gli dà un vettore.

In [ ]:
FT_P = FastText(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1, min_n=3, max_n=5)

def vector_de(modelo, palabra):
    esta = None      # TODO: palabra è in modelo.wv.key_to_index?
    vector = None    # TODO: modelo.wv[palabra]
    return esta, vector

# Controllo automatico
_esta, _vec = vector_de(FT_P, "caballeroo")
assert _esta is not None and _vec is not None, "Completa le righe contrassegnate con TODO"
assert _esta is False and _vec.shape == (30,), "il refuso non è nel vocabolario, ma fastText gli dà un vettore di 30 dimensioni"
assert vector_de(FT_P, "caballero")[0] is True
_w2v = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1)
try:
    _w2v.wv["caballeroo"]
    _lanza = False
except KeyError:
    _lanza = True
assert _lanza, "Word2Vec dovrebbe sollevare KeyError"
_cos_e = float(np.dot(_vec, FT_P.wv["caballero"]) / (np.linalg.norm(_vec) * np.linalg.norm(FT_P.wv["caballero"])))
assert -1 <= _cos_e <= 1
print("Esercizio 5 corretto ✔", "cos(caballeroo, caballero) =", round(_cos_e, 3))

---
## Sfida finale — Più testo, meno rumore?

Con **le tue** funzioni `entrenar_p` (esercizio 3) e `solape_p` (esercizio 4) e con `frecuentes(frases, modelo, n)`:

1. Completa `solape_semillas(frases, n=100)`: addestra due modelli skip-gram (`window=5`) con i semi 1 e 2, prendi le `n` parole più frequenti (`frecuentes`, calcolate sul modello del seme 1) e restituisci il `solape_p` a 10 vicini.
2. Calcolalo per tre dimensioni: l'estratto piccolo (`PEQUENO`), il **10 %** e il **50 %** delle frasi del corpus spagnolo grande (`GRANDE`, quattro opere, ~758 000 parole). Salva i risultati in `res_tamano` (chiavi `"pequeno"`, `"10%"`, `"50%"`).
3. Scrivi sotto le tue conclusioni: il solape sale con il testo? Basta per fidarsi di un vicino isolato?

Ci vogliono circa 15-20 secondi. *(Esempio illustrativo: con altri corpus e parametri le cifre cambiano.)*

In [ ]:
with open(cargar_datos(RUTA_GRANDE), encoding="utf-8") as f:
    GRANDE = frases_de_texto(f.read())

def solape_semillas(frases, n=100):
    return None    # TODO: addestra con entrenar_p (sg=1, window=5, semi 1 e 2) e restituisci solape_p su frecuentes(frases, m1, n) a 10 vicini

res_tamano = {}
# TODO: riempi res_tamano["pequeno"], res_tamano["10%"] e res_tamano["50%"] con solape_semillas (le frazioni sono le prime frasi di GRANDE)

# Controllo automatico
assert len(res_tamano) == 3 and all(v is not None for v in res_tamano.values()), "Completa le righe contrassegnate con TODO"
assert all(0 <= v <= 1 for v in res_tamano.values()) and set(res_tamano) == {"pequeno", "10%", "50%"}
assert res_tamano["pequeno"] < res_tamano["10%"] and res_tamano["pequeno"] < res_tamano["50%"], "con pochissimo testo il solape dovrebbe essere il più basso"
print("Sfida finale corretta ✔")
print({k: round(v, 3) for k, v in res_tamano.items()})

**Le tue conclusioni (scrivi 3-4 frasi):** perché il coseno ignora la lunghezza del vettore? Cosa cambia tra un vettore sparso di co-occorrenze e un embedding denso? Perché servono `workers=1` e un seme fisso per riprodurre un modello? Come cambia il solape con la dimensione del corpus e cosa implica per interpretare un vicino isolato? Cosa guadagna e cosa non guadagna fastText con un refuso?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def coseno_p(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Controllo automatico
_c = coseno_p([1, 2, 2], [2, 1, 2])
assert _c is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_c, 8 / 9), "a mano: 8/9 ≈ 0,889"
assert np.isclose(coseno_p([3, 4], [30, 40]), 1.0) and np.isclose(coseno_p([3, 4], [-4, 3]), 0.0) and np.isclose(coseno_p([3, 4], [-3, -4]), -1.0)
assert np.isclose(coseno_p([1, 2, 2], [2, 1, 2]), coseno_p([2, 4, 4], [1, 1, 1] + np.array([1, 0, 1])))   # non dipende dalla lunghezza
print("Esercizio 1 corretto ✔", round(_c, 3))

### Soluzione dell'esercizio 2

In [ ]:
def matriz_coocurrencia(frases, vocab, ventana=1):
    indice = {p: i for i, p in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)), dtype=int)
    for frase in frases:
        for i, p in enumerate(frase):
            for j in range(max(0, i - ventana), min(len(frase), i + ventana + 1)):
                if j != i and p in indice and frase[j] in indice:
                    M[indice[p], indice[frase[j]]] += 1
    return M

# Controllo automatico
def _cos(a, b):
    return a @ b / np.linalg.norm(a) / np.linalg.norm(b)

_M1 = matriz_coocurrencia(FRASES_P, VOCAB_P, 1)
_ix = {p: i for i, p in enumerate(VOCAB_P)}
assert _M1.sum() > 0, "Completa le righe contrassegnate con TODO"
assert _M1.shape == (9, 9) and (_M1 == _M1.T).all(), "la matrice deve essere quadrata e simmetrica"
assert [int(_M1[_ix["rey"], _ix[w]]) for w in VOCAB_P[4:5] + ["ama", "al", "reina"]] == [1, 1, 1, 0]      # il calcolo a mano
assert np.isclose(_cos(_M1[_ix["rey"]], _M1[_ix["pastor"]]), 1.0)          # stessi contesti con finestra 1
assert np.isclose(_cos(_M1[_ix["rey"]], _M1[_ix["reina"]]), 1 / np.sqrt(15))
_M2 = matriz_coocurrencia(FRASES_P, VOCAB_P, 2)
assert _M2.sum() > _M1.sum() and np.isclose(_cos(_M2[_ix["rey"]], _M2[_ix["reina"]]), 4 / 7)   # finestra maggiore: più co-occorrenze
print("Esercizio 2 corretto ✔", int(_M1.sum()), int(_M2.sum()))

### Soluzione dell'esercizio 3

In [ ]:
def entrenar_p(frases, sg, window, seed):
    return Word2Vec(frases, vector_size=30, window=window, min_count=3, sg=sg, epochs=5, workers=1, seed=seed)

# Controllo automatico
_a = entrenar_p(PEQUENO, 1, 5, 1)
assert _a is not None, "Completa le righe contrassegnate con TODO"
_b = entrenar_p(PEQUENO, 1, 5, 1)
_c = entrenar_p(PEQUENO, 0, 3, 2)
assert _a.wv.vector_size == 30 and _a.sg == 1 and _c.sg == 0 and _c.window == 3, "rivedi vector_size, sg e window"
assert _a.wv.key_to_index == _b.wv.key_to_index and np.allclose(_a.wv.vectors, _b.wv.vectors), "stessi dati e stesso seme devono dare lo stesso modello (workers=1)"
assert not np.allclose(_a.wv.vectors, entrenar_p(PEQUENO, 1, 5, 2).wv.vectors), "un altro seme deve dare altri vettori"
print("Esercizio 3 corretto ✔", "vocabolario:", len(_a.wv))

### Soluzione dell'esercizio 4

In [ ]:
def solape_p(m1, m2, palabras, k=10):
    valores = []
    for w in palabras:
        valores.append(len(set(vecinos(m1, w, k)) & set(vecinos(m2, w, k))) / k)
    return float(np.mean(valores)) if valores else None

# Controllo automatico
class _Falso:                                     # modello giocattolo che restituisce vicini fissi
    def __init__(self, lista):
        self.wv = self
        self._lista = lista
    def most_similar(self, palabra, topn=10):
        return [(w, 1.0) for w in self._lista[:topn]]

_f1, _f2 = _Falso(["a", "b", "c", "d"]), _Falso(["a", "b", "x", "y"])
_r = solape_p(_f1, _f2, ["p"], 4)
assert _r is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_r, 0.5) and np.isclose(solape_p(_f1, _f1, ["p", "q"], 4), 1.0), "a mano: 2 su 4 = 0,5"
_m1 = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1)
_m2 = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=2)
_frec = frecuentes(PEQUENO, _m1, 50)
_mismo, _otro = solape_p(_m1, _m1, _frec, 10), solape_p(_m1, _m2, _frec, 10)
assert np.isclose(_mismo, 1.0) and 0 <= _otro < 1
print("Esercizio 4 corretto ✔", "solape tra semi nell'estratto piccolo:", round(_otro, 3))

### Soluzione dell'esercizio 5

In [ ]:
FT_P = FastText(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1, min_n=3, max_n=5)

def vector_de(modelo, palabra):
    esta = palabra in modelo.wv.key_to_index
    vector = modelo.wv[palabra]
    return esta, vector

# Controllo automatico
_esta, _vec = vector_de(FT_P, "caballeroo")
assert _esta is not None and _vec is not None, "Completa le righe contrassegnate con TODO"
assert _esta is False and _vec.shape == (30,), "il refuso non è nel vocabolario, ma fastText gli dà un vettore di 30 dimensioni"
assert vector_de(FT_P, "caballero")[0] is True
_w2v = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1)
try:
    _w2v.wv["caballeroo"]
    _lanza = False
except KeyError:
    _lanza = True
assert _lanza, "Word2Vec dovrebbe sollevare KeyError"
_cos_e = float(np.dot(_vec, FT_P.wv["caballero"]) / (np.linalg.norm(_vec) * np.linalg.norm(FT_P.wv["caballero"])))
assert -1 <= _cos_e <= 1
print("Esercizio 5 corretto ✔", "cos(caballeroo, caballero) =", round(_cos_e, 3))

### Soluzione della sfida finale

In [ ]:
with open(cargar_datos(RUTA_GRANDE), encoding="utf-8") as f:
    GRANDE = frases_de_texto(f.read())

def solape_semillas(frases, n=100):
    m1, m2 = entrenar_p(frases, 1, 5, 1), entrenar_p(frases, 1, 5, 2)
    return solape_p(m1, m2, frecuentes(frases, m1, n), 10)

res_tamano = {"pequeno": solape_semillas(PEQUENO),
              "10%": solape_semillas(GRANDE[: len(GRANDE) // 10]),
              "50%": solape_semillas(GRANDE[: len(GRANDE) // 2])}

# Controllo automatico
assert len(res_tamano) == 3 and all(v is not None for v in res_tamano.values()), "Completa le righe contrassegnate con TODO"
assert all(0 <= v <= 1 for v in res_tamano.values()) and set(res_tamano) == {"pequeno", "10%", "50%"}
assert res_tamano["pequeno"] < res_tamano["10%"] and res_tamano["pequeno"] < res_tamano["50%"], "con pochissimo testo il solape dovrebbe essere il più basso"
print("Sfida finale corretta ✔")
print({k: round(v, 3) for k, v in res_tamano.items()})

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto non ti è rimasto chiaro (coseno, co-occorrenza e vettori sparsi, skip-gram contro CBOW, rumore tra semi, fastText e n-grammi)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*